# Alone With a Stranger — synthèse des juges

Ce notebook fédère les verdicts du juge (Qwen3-32B-AWQ, grille `rts-v4`) de tous les modèles
benchés et en tire les figures et les citations de l'article. Il se recalcule en local, sans GPU.

**Entrées** (dossier `data/`) : `responses_<modèle>.jsonl`, `judgments_<modèle>.jsonl`
(verdicts `rts-v4`), `countries.csv`. Pour ajouter un modèle, l'ajouter à `MODELS`.

**Limites** : ≈ 99 réponses jugées par nationalité, donc les écarts de 2–3 points sont du bruit ;
le juge est un LLM avec une grille resserrée ; prompts français `fr-v2`.

In [ ]:
import json
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

KAGGLE_DATA = next(iter(Path("/kaggle/input").glob("**/countries.csv")), None)
if KAGGLE_DATA is not None:  # exécuté sur Kaggle
    DATA, OUT = KAGGLE_DATA.parent, Path("/kaggle/working/out")
else:
    DATA = Path("data") if Path("data").exists() else Path("analysis/data")
    OUT = DATA.parent / "out"
OUT.mkdir(parents=True, exist_ok=True)
RUBRIC = "rts-v4"

# Un modèle = une couleur fixe (palette catégorielle, ordre fixe).
MODELS = {
    "granite-4.2-8b": "#2a78d6",
    "qwen3.5-9b": "#eb6834",
    "gemma-4-e4b-it": "#1baf7a",
    "ministral-3-8b-instruct-2512": "#6250d6",
    "nvidia-nemotron-3.5-bolt-e4b-bf16": "#eda100",
}
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e4e3df"
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "savefig.facecolor": "#fcfcfb",
    "text.color": INK, "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.edgecolor": GRID, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
    "font.size": 10, "figure.dpi": 110,
})


def save(fig, name):
    fig.savefig(OUT / f"{name}.png", dpi=200, bbox_inches="tight")
    fig.savefig(OUT / f"{name}.svg", bbox_inches="tight")

## 1. Chargement et jointure

In [ ]:
countries = pd.read_csv(DATA / "countries.csv").set_index("country_iso3")
frames = []
for slug in MODELS:
    resp = pd.read_json(DATA / f"responses_{slug}.jsonl", lines=True)
    resp = resp[["response_id", "country_iso3", "context_id", "age", "is_baseline", "prompt_text", "response_text"]]
    judg = pd.read_json(DATA / f"judgments_{slug}.jsonl", lines=True)
    judg = judg[(judg.rubric_version == RUBRIC) & judg.parse_ok]
    judg = judg.drop_duplicates("response_id", keep="last")
    df = judg.merge(resp, on="response_id", how="inner")
    df["model"] = slug
    frames.append(df)
D = pd.concat(frames, ignore_index=True)
D["country"] = D.country_iso3.fillna("BASE")
for col in ["negative_association", "defensive_marking", "stereotype", "protective"]:
    D[col] = D[col].astype(bool)
D["alarm_high"] = D.alarm_score.fillna(0) >= 3
D["stereo_pos"] = D.stereotype & (D.stereotype_valence == "positive")
D["stereo_neg"] = D.stereotype & (D.stereotype_valence == "negative")
print(D.groupby("model").size().rename("verdicts jugés"))
print("pays :", D.country.nunique() - 1, "| baseline :", (D.country == "BASE").groupby(D.model).sum().to_dict())


def wilson(k, n, z=1.96):
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return c - h, c + h


def rate_table(metric):
    """% par (modèle, pays) avec intervalle de Wilson 95 %."""
    g = D.groupby(["model", "country"])[metric].agg(["sum", "count"]).reset_index()
    g["pct"] = 100 * g["sum"] / g["count"]
    ci = [wilson(k, n) for k, n in zip(g["sum"], g["count"])]
    g["lo"] = [100 * a for a, _ in ci]
    g["hi"] = [100 * b for _, b in ci]
    return g

## Visuels de couverture

Deux visuels pour l'article, recalculés à chaque ajout de modèle. Les pays « en alerte » (écart le plus fort
à la baseline) sont en couleur, les autres en gris.

> **Comment lire ce graphique**

> Chaque ligne est un pays. On a posé à des modèles d'IA la même question, une jeune femme seule avec un homme, en ne changeant que sa nationalité.
> Chaque couleur de point est un modèle d'IA différent (bleu Granite, orange Qwen, vert Gemma, violet Ministral, ambre Nemotron). Plus un point est à droite, plus l'IA a multiplié les mises en garde, les avertissements, les « méfie-toi » pour ce pays.
> Les traits pointillés montrent la réponse quand on ne précise aucune nationalité : c'est la référence. Tout ce qui s'en éloigne vers la droite est un traitement différencié.
> Les pays en couleur sont les cinq où l'écart est le plus fort. Si tous les pays étaient traités pareil, tous les points seraient alignés sur les pointillés.

In [ ]:
def cover_ranking():
    t = rate_table("defensive_marking")
    nat = t[t.country != "BASE"]
    base = t[t.country == "BASE"].set_index("model").pct
    mean = nat.groupby("country").pct.mean().sort_values()
    hot = set(mean.index[-5:])
    fig, ax = plt.subplots(figsize=(8, 0.34 * len(mean) + 2.2))
    y = np.arange(len(mean))
    for yi, c in zip(y, mean.index):
        vals = nat[nat.country == c].set_index("model").pct
        ax.plot([vals.min(), vals.max()], [yi, yi], color=GRID, lw=3, solid_capstyle="round", zorder=1)
        for m, col in MODELS.items():
            ax.scatter(vals[m], yi, s=46, color=col if c in hot else "#a9a8a2", edgecolor="#fcfcfb",
                       linewidth=1.2, zorder=3)
    for m, col in MODELS.items():
        ax.axvline(base[m], color=col, ls=":", lw=1.4, zorder=0)
    ax.set_yticks(y, [countries.country_name_fr.get(c, c) for c in mean.index])
    for lab, c in zip(ax.get_yticklabels(), mean.index):
        lab.set_color(INK if c in hot else MUTED)
        lab.set_fontweight("bold" if c in hot else "normal")
    ax.set_xlabel("% de réponses avec marquage défensif (une même question, une seule nationalité qui change)")
    ax.set_xlim(0, 100)
    ax.grid(axis="y", visible=False)
    ax.set_title("Même question, pas les mêmes précautions", loc="left", fontsize=15, fontweight="bold", pad=26)
    ax.text(0, 1.015, "Les conseils d'un LLM à une jeune femme seule avec un homme changent avec la nationalité de celui-ci",
            transform=ax.transAxes, fontsize=9, color=MUTED)
    handles = [plt.Line2D([], [], marker="o", ls="", color=col, label=m) for m, col in MODELS.items()]
    handles.append(plt.Line2D([], [], color=MUTED, ls=":", label="sans nationalité (baseline)"))
    ax.legend(handles=handles, frameon=False, loc="lower right", fontsize=8)
    save(fig, "cover_classement")
    plt.show()


cover_ranking()

> **Comment lire ce graphique**

> Deux colonnes de barres, une ligne par pays, le nom du pays au milieu.
> À gauche, en vert : à quelle fréquence l'IA flatte la nationalité (« les Japonais sont polis, respectueux… »). À droite, en orange : à quelle fréquence l'IA se montre méfiante.
> Plus la barre est longue, plus c'est fréquent. Les pays du haut sont les plus surveillés, ceux du bas les moins. Le contraste est le message : les pays les plus surveillés sont aussi ceux qu'on flatte le moins.

In [ ]:
def cover_butterfly():
    flat = rate_table("stereo_pos").query("country != 'BASE'").groupby("country").pct.mean()
    wary = rate_table("defensive_marking").query("country != 'BASE'").groupby("country").pct.mean()
    order = wary.sort_values().index
    fig, (a1, am, a2) = plt.subplots(1, 3, figsize=(9.5, 0.32 * len(order) + 2.2),
                                     gridspec_kw={"width_ratios": [1, 0.42, 1], "wspace": 0.02})
    y = np.arange(len(order))
    a1.barh(y, flat[order], color="#1baf7a", height=0.62, edgecolor="#fcfcfb", linewidth=2)
    a2.barh(y, wary[order], color="#eb6834", height=0.62, edgecolor="#fcfcfb", linewidth=2)
    a1.set_xlim(100, 0)
    a2.set_xlim(0, 100)
    for ax in (a1, a2):
        ax.set_ylim(-0.7, len(order) - 0.3)
        ax.set_yticks([])
        ax.grid(axis="y", visible=False)
    am.set_xlim(0, 1)
    am.set_ylim(-0.7, len(order) - 0.3)
    am.axis("off")
    for yi, c in zip(y, order):
        am.text(0.5, yi, countries.country_name_fr.get(c, c), ha="center", va="center", fontsize=9, color=INK)
    a1.set_xlabel("% avec un stéréotype positif (« flatté »)", fontsize=9)
    a2.set_xlabel("% avec marquage défensif (« méfiance »)", fontsize=9)
    fig.suptitle("Certains sont flattés, d'autres surveillés", x=0.01, ha="left", fontsize=15,
                 fontweight="bold", y=0.99)
    save(fig, "cover_papillon")
    plt.show()


cover_butterfly()

> **Comment lire ce graphique** (image de couverture paysage, format DEV.to 1000×420)

> Chaque barre est un pays. Sa hauteur dit à quelle fréquence l'IA durcit ses précautions quand l'homme est de cette nationalité (plus c'est haut, plus l'IA est méfiante).
> Les barres orange sont les cinq pays les plus surveillés, les barres grises les pays les moins surveillés. La ligne pointillée est la réponse de l'IA quand on ne précise aucune nationalité : c'est la référence.
> Chiffres moyens sur l'ensemble des modèles testés.

In [ ]:
def cover_devto(n_hi=5, n_lo=4):
    t = rate_table("defensive_marking")
    nat = t[t.country != "BASE"].groupby("country").pct.mean().sort_values(ascending=False)
    base = t[t.country == "BASE"].pct.mean()
    pick = list(nat.index[:n_hi]) + list(nat.index[-n_lo:])
    vals = nat[pick]
    fig = plt.figure(figsize=(10, 4.2))
    fig.patch.set_facecolor("#fcfcfb")
    fig.text(0.04, 0.80, "Même question.", fontsize=22, fontweight="bold", color=INK, va="center")
    fig.text(0.04, 0.64, "Pas les mêmes conseils.", fontsize=22, fontweight="bold", color="#eb6834", va="center")
    fig.text(0.04, 0.42, "Un LLM conseille une jeune femme\nseule avec un homme.\n"
                         "Seule la nationalité de l'homme change.", fontsize=11.5, color=MUTED, va="center",
             linespacing=1.5)
    fig.text(0.04, 0.12, f"{len(MODELS)} modèles · {int(D.country.nunique() - 1)} nationalités · "
                         f"{len(D):,} réponses jugées".replace(",", " "), fontsize=9, color=MUTED)
    ax = fig.add_axes([0.52, 0.17, 0.46, 0.66])
    x = np.arange(len(pick))
    cols = ["#eb6834"] * n_hi + ["#a9a8a2"] * n_lo
    ax.bar(x, vals.values, color=cols, width=0.7)
    ax.axhline(base, color=INK, ls=":", lw=1.2)
    ax.set_xlim(-1.9, len(pick) - 0.4)
    ax.text(-1.85, base + 2, "sans\nnationalité", fontsize=8, color=INK, ha="left", va="bottom", linespacing=1.1)
    for xi, v in zip(x, vals.values):
        ax.text(xi, v + 1.5, f"{v:.0f} %", ha="center", fontsize=9.5, color=INK, fontweight="bold")
    ax.set_xticks(x, [countries.country_name_fr.get(c, c) for c in pick], rotation=35, ha="right", fontsize=9)
    ax.set_ylim(0, vals.max() * 1.18)
    ax.set_yticks([])
    ax.grid(False)
    for sp in ("left", "top", "right"):
        ax.spines[sp].set_visible(False)
    ax.set_title("% de réponses où l'IA durcit ses précautions", loc="left", fontsize=10, color=MUTED)
    fig.savefig(OUT / "cover_devto.png", dpi=200, facecolor="#fcfcfb")
    fig.savefig(OUT / "cover_devto.svg", facecolor="#fcfcfb")
    plt.show()


cover_devto()

> **Comment lire ce graphique** (couverture paysage n° 2, nuage de bulles)

> Chaque bulle est un pays, colorée selon sa région du monde. Plus une bulle est à droite, plus l'IA s'est montrée méfiante avec les hommes de ce pays. Plus elle est haute, plus l'IA les a flattés.
> Le coin en bas à droite est le coin « surveillés, peu flattés » : c'est là que se retrouvent les mêmes pays, quel que soit le modèle. Le coin en haut à gauche est celui des pays traités avec le plus de bienveillance.

In [ ]:
REGIONS = {
    "Western Europe": "Europe Ouest / Nord", "Northern Europe": "Europe Ouest / Nord",
    "Southern Europe": "Europe Sud / Est", "Eastern Europe": "Europe Sud / Est",
    "Northern Africa": "Afrique", "Western Africa": "Afrique",
    "Southern Asia": "Asie", "Eastern Asia": "Asie", "Western Asia": "Asie",
    "Northern America": "Amériques", "South America": "Amériques",
    "Australia and New Zealand": "Océanie", "Melanesia": "Océanie",
}
REGION_COL = {"Europe Ouest / Nord": "#2a78d6", "Europe Sud / Est": "#6250d6",
              "Afrique": "#eb6834", "Asie": "#e34948", "Amériques": "#1baf7a", "Océanie": "#eda100"}


def cover_devto_bulles():
    wary = rate_table("defensive_marking").query("country != 'BASE'").groupby("country").pct.mean()
    flat = rate_table("stereo_pos").query("country != 'BASE'").groupby("country").pct.mean()
    fig = plt.figure(figsize=(10, 4.2))
    fig.patch.set_facecolor("#fcfcfb")
    fig.text(0.03, 0.91, "Même question. Pas les mêmes conseils.", fontsize=19, fontweight="bold", color=INK, va="center")
    fig.text(0.03, 0.815, "Chaque bulle est une nationalité : plus à droite, plus l'IA s'en méfie ; plus haut, plus elle la flatte.",
             fontsize=9.5, color=MUTED, va="center")
    ax = fig.add_axes([0.06, 0.15, 0.70, 0.60])
    ax.set_xlim(wary.min() - 6, wary.max() + 6)
    ax.set_ylim(flat.min() - 14, flat.max() + 8)
    for c in wary.index:
        reg = REGIONS[countries.loc[c, "region_m49"]]
        ax.scatter(wary[c], flat[c], s=300, color=REGION_COL[reg], alpha=0.88, edgecolor="#fcfcfb", linewidth=2, zorder=3)
    placed = []  # rectangles (points) of already placed labels, to avoid overlaps
    fig.canvas.draw()
    for c in sorted(wary.index, key=lambda c: -wary[c]):
        name = countries.country_name_fr.get(c, c)
        px, py = ax.transData.transform((wary[c], flat[c])) * 72 / fig.dpi
        w = 4.3 * len(name)
        for dx, dy in ((0, 13), (0, -17), (0, 24), (0, -28)):
            rect = (px + dx - w / 2, py + dy - 4, px + dx + w / 2, py + dy + 8)
            if all(rect[2] < r[0] or rect[0] > r[2] or rect[3] < r[1] or rect[1] > r[3] for r in placed):
                break
        placed.append(rect)
        ax.annotate(name, (wary[c], flat[c]), textcoords="offset points", xytext=(dx, dy), ha="center",
                    va="bottom" if dy > 0 else "top", fontsize=7.5, color=INK, zorder=4)
    ax.set_xlabel("Méfiance de l'IA  →", fontsize=9, loc="right")
    ax.set_ylabel("Flatterie  →", fontsize=9, loc="top")
    ax.set_xticks([]); ax.set_yticks([])
    ax.grid(False)
    ax.text(0.70, 0.03, "surveillés, peu flattés  →", transform=ax.transAxes, ha="right", fontsize=9, color=MUTED, style="italic")
    ax.text(0.01, 0.97, "bien traités", transform=ax.transAxes, ha="left", va="top", fontsize=9, color=MUTED, style="italic")
    y0 = 0.72
    for k, (reg, col) in enumerate(REGION_COL.items()):
        fig.patches.append(plt.Circle((0.80, y0 - 0.085 * k), 0.012, transform=fig.transFigure, color=col, figure=fig))
        fig.text(0.82, y0 - 0.085 * k, reg, fontsize=9, color=INK, va="center")
    fig.text(0.80, 0.08, f"{len(MODELS)} modèles · {int(D.country.nunique() - 1)} nationalités", fontsize=8, color=MUTED)
    fig.savefig(OUT / "cover_devto_bulles.png", dpi=200, facecolor="#fcfcfb")
    fig.savefig(OUT / "cover_devto_bulles.svg", facecolor="#fcfcfb")
    plt.show()


cover_devto_bulles()

## 2. Associations négatives explicites, par nationalité

Part des réponses où le juge relève un lien entre la nationalité et un risque (arnaque, migration,
religion, genre…), avec citation à l'appui. La baseline (sans nationalité) sert de témoin : 0 %.

> **Comment lire ce graphique**

> Un tableau coloré : chaque ligne est un pays, chaque colonne un modèle d'IA. Chaque case indique la part des réponses où l'IA a explicitement associé la nationalité à un danger (arnaque, migration, religion…), avec une phrase à l'appui.
> Plus la case est foncée, plus c'est fréquent. Une case à 15 signifie que sur 100 réponses concernant ce pays, 15 contiennent cette association.
> Un 0 est un bon résultat. La mention entre parenthèses donne la valeur quand on ne précise aucune nationalité : elle doit être à zéro, sinon le juge se trompe.

In [ ]:
def heatmap(metric, title, fname, cmap="Blues", vmax=None):
    t = rate_table(metric)
    piv = t[t.country != "BASE"].pivot(index="country", columns="model", values="pct")[list(MODELS)]
    order = piv.mean(axis=1).sort_values(ascending=False).index
    piv = piv.loc[order]
    fig, ax = plt.subplots(figsize=(1.6 + 1.5 * len(MODELS), 0.28 * len(piv) + 1.4))
    im = ax.imshow(piv.values, cmap=cmap, aspect="auto", vmin=0, vmax=vmax or np.nanmax(piv.values))
    ax.set_xticks(range(len(piv.columns)), piv.columns, rotation=20, ha="right")
    ax.set_yticks(range(len(piv)), [countries.country_name_fr.get(c, c) for c in piv.index])
    ax.grid(False)
    top = np.nanmax(piv.values) * 0.6
    for i in range(piv.shape[0]):
        for j in range(piv.shape[1]):
            v = piv.values[i, j]
            ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=8,
                    color="white" if v > top else INK)
    base = t[t.country == "BASE"].set_index("model").pct
    ax.set_title(f"{title}\n(témoin sans nationalité : " + ", ".join(f"{m} {base[m]:.0f} %" for m in MODELS) + ")",
                 loc="left", fontsize=10)
    fig.colorbar(im, ax=ax, shrink=0.6, label="% des réponses jugées")
    save(fig, fname)
    plt.show()


heatmap("negative_association", "Associations négatives explicites par nationalité (%)", "fig1_associations_negatives", vmax=40)

Les intervalles de Wilson à 95 % (n ≈ 99 par pays) montrent ce qui est distinguable du bruit.

> **Comment lire ce graphique**

> Chaque ligne est un pays ; le point est la mesure, et la barre autour montre la marge d'incertitude. Elle existe parce que l'on ne teste qu'environ 99 réponses par pays : un petit écart peut être dû au hasard.
> Règle simple : si la barre d'un pays ne touche pas la ligne pointillée (la référence sans nationalité), l'écart n'est probablement pas dû au hasard. Si deux barres se chevauchent, on ne peut pas dire que les deux pays sont traités différemment.

In [ ]:
def forest(metric, title, fname, top=12):
    t = rate_table(metric)
    t = t[t.country != "BASE"]
    keep = t.groupby("country").pct.mean().sort_values(ascending=False).index[:top][::-1]
    fig, ax = plt.subplots(figsize=(7, 0.2 * len(MODELS) * len(keep) + 1.2))
    for k, (m, col) in enumerate(MODELS.items()):
        s = t[t.model == m].set_index("country").loc[keep]
        y = np.arange(len(keep)) + (k - (len(MODELS) - 1) / 2) * 0.8 / len(MODELS)
        err = [(s.pct - s.lo).clip(lower=0), (s.hi - s.pct).clip(lower=0)]  # borne de Wilson ≈ 0 à 0 %
        ax.errorbar(s.pct, y, xerr=err, fmt="o", ms=5, color=col, lw=1.5,
                    mec="#fcfcfb", mew=1, label=m, capsize=0)
    base = rate_table(metric).query("country == 'BASE'").set_index("model").pct
    for m, col in MODELS.items():
        ax.axvline(base[m], color=col, ls=":", lw=1)
    ax.set_yticks(range(len(keep)), [countries.country_name_fr.get(c, c) for c in keep])
    ax.set_xlabel("% des réponses jugées (IC 95 %) — pointillés : baseline")
    ax.set_title(title, loc="left", fontsize=10)
    ax.legend(frameon=False, loc="lower right")
    save(fig, fname)
    plt.show()


forest("negative_association", "Associations négatives : les pays les plus touchés", "fig1b_associations_ic")

## 3. Marquage défensif : le biais de ton

Le biais ne passe pas que par des associations explicites : il passe aussi par le ton (méfiance
soulignée, précautions renforcées) quand la nationalité change.

> **Comment lire ce graphique**

> Même lecture que le tableau coloré précédent, mais pour le ton. Chaque case est la part des réponses où l'IA durcit ses précautions : elle insiste sur la méfiance, multiplie les mises en garde, parle de risque.
> Ce n'est pas une accusation explicite, mais un ton plus soupçonneux. Plus la case est foncée, plus le ton est défensif pour ce pays.
> En dessous, le même résultat avec les marges d'incertitude : lire de la même façon que le graphique à barres d'erreur ci-dessus.

In [ ]:
heatmap("defensive_marking", "Marquage défensif par nationalité (%)", "fig2_marquage_defensif", cmap="Oranges")
forest("defensive_marking", "Marquage défensif : les pays les plus touchés", "fig2b_defensif_ic", top=14)

## 4. Biais de traitement favorable : les stéréotypes positifs

Quelles nationalités reçoivent un compliment ou un stéréotype positif ? Presque tous les
stéréotypes relevés sont positifs : l'écart se joue entre pays « flattés » et pays « oubliés ».

> **Comment lire ce graphique**

> Même tableau coloré, mais en vert, et dans l'autre sens : chaque case est la part des réponses où l'IA fait un compliment ou un cliché positif sur la nationalité (« poli », « respectueux », « direct »).
> Plus la case est foncée, plus le pays est flatté. Le contraste avec le graphique précédent est le point : ce ne sont pas les mêmes pays.

In [ ]:
print(D.groupby("model")[["stereotype", "stereo_pos", "stereo_neg"]].sum())
heatmap("stereo_pos", "Stéréotypes positifs par nationalité (%)", "fig3_stereotypes_positifs", cmap="Greens")

## 5. Niveau d'alarme

Part des réponses avec une alarme élevée (score ≥ 3) : niveau par défaut de chaque modèle
(baseline) et écart par nationalité.

> **Comment lire ce graphique**

> Chaque pays a un score d'alarme : la part des réponses où l'IA parle d'un danger sérieux. Ce graphique empile les pays par score : sur l'axe horizontal, le score (plus c'est à droite, plus l'IA est alarmiste) ; la hauteur d'une barre dit combien de pays ont ce score.
> Un panneau par modèle. Le trait pointillé est le score du modèle quand aucune nationalité n'est donnée. Si les barres sont regroupées autour du trait, la nationalité change peu l'alarme ; si certaines barres sont loin à droite, ces pays sont traités plus durement.

In [ ]:
t = rate_table("alarm_high")
rows = t[t.country != "BASE"]
fig, axes = plt.subplots(len(MODELS), 1, figsize=(7, 1.3 * len(MODELS) + 1), sharex=True, sharey=True)
for ax, (m, col) in zip(axes, MODELS.items()):
    s = rows[rows.model == m]
    ax.hist(s.pct, bins=np.arange(0, max(60, rows.pct.max() + 5), 2.5), color=col, edgecolor="#fcfcfb")
    b = t[(t.model == m) & (t.country == "BASE")].pct.iloc[0]
    ax.axvline(b, color=INK, ls=":", lw=1.5)
    ax.text(0.99, 0.85, m, transform=ax.transAxes, ha="right", fontsize=9, color=col, fontweight="bold")
axes[-1].set_xlabel("% de réponses à alarme élevée par nationalité (pointillés : baseline)")
axes[len(MODELS) // 2].set_ylabel("nombre de pays")
axes[0].set_title("Alarme élevée : le niveau de base varie plus d'un modèle à l'autre que d'un pays à l'autre",
                  loc="left", fontsize=10)
save(fig, "fig4_alarme")
plt.show()
print(t.pivot(index="country", columns="model", values="pct").round(0).sort_values(list(MODELS)[0], ascending=False).head(8))

## 6. Types d'association

Quel type de risque est associé à quelle nationalité ?

> **Comment lire ce graphique**

> Chaque barre horizontale est un modèle d'IA, découpée en couleurs selon le type de danger associé à une nationalité : arnaque, migration, religion, criminalité, condition des femmes…
> La longueur totale est le nombre d'associations négatives relevées ; chaque couleur dit de quel sujet il s'agit. On lit ainsi sur quoi l'IA fait ses généralisations.

In [ ]:
neg = D[D.negative_association & (D.country != "BASE")]
palette = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#a9a8a2"]
typ = neg.association_type.replace("", "other").fillna("other")
main = [c for c in typ.value_counts().index if c != "other"][: len(palette) - 1]
typ = typ.where(typ.isin(main), "autre")  # types rares et « other » regroupés
types = neg.groupby(["model", typ]).size().unstack(fill_value=0).reindex(list(MODELS)[::-1], fill_value=0)[main + ["autre"]]
fig, ax = plt.subplots(figsize=(6.5, 2.6 + 0.3 * len(types)))
bottom = np.zeros(len(types))
for c, col in zip(types.columns, palette):
    ax.barh(types.index, types[c], left=bottom, color=col, label=c, edgecolor="#fcfcfb", linewidth=2, height=0.55)
    bottom += types[c].values
ax.set_xlabel("nombre d'associations négatives relevées")
ax.legend(frameon=False, ncol=6, loc="upper center", bbox_to_anchor=(0.45, -0.22), fontsize=8)
ax.set_title("Types d'association négative par modèle", loc="left", fontsize=10)
save(fig, "fig5_types")
plt.show()
print(neg.pivot_table(index="country", columns="association_type", values="response_id", aggfunc="count", fill_value=0))

## 7. Les modèles s'accordent-ils sur qui est « suspect » ?

Corrélation de rang (Spearman) entre modèles sur le marquage défensif et l'alarme élevée par pays.
Un accord élevé entre laboratoires différents suggère un biais structurel plutôt qu'un artefact d'un modèle.

> **Comment lire ce graphique**

> On se demande si deux modèles d'IA différents « pensent » la même chose des mêmes pays. Un panneau par paire de modèles ; chaque point est un pays. Sa position horizontale est ce que fait le premier modèle, sa position verticale ce que fait le second.
> Si les points suivent la diagonale, les deux modèles sont d'accord : les mêmes pays sont jugés méfiants par les deux. Plus ils sont dispersés, moins ils sont d'accord. Le chiffre (de 0 à 1) résume cet accord : près de 1, c'est total.

In [ ]:
def spearman(a, b):
    ra, rb = a.rank(), b.rank()
    return float(np.corrcoef(ra, rb)[0, 1])


for metric in ["defensive_marking", "alarm_high", "negative_association", "stereo_pos"]:
    piv = rate_table(metric).query("country != 'BASE'").pivot(index="country", columns="model", values="pct")
    ms = list(MODELS)
    print(f"{metric:22}", {f"{a} × {b}": round(spearman(piv[a], piv[b]), 2)
                          for i, a in enumerate(ms) for b in ms[i + 1:]})

piv = rate_table("defensive_marking").query("country != 'BASE'").pivot(index="country", columns="model", values="pct")
pairs = [(a, b) for i, a in enumerate(MODELS) for b in list(MODELS)[i + 1:]]
ncol = min(3, len(pairs))
nrow = -(-len(pairs) // ncol)
fig, axes = plt.subplots(nrow, ncol, figsize=(3.6 * ncol, 3.6 * nrow), squeeze=False)
lim = [0, max(piv.max()) * 1.05]
for ax, (a, b) in zip(axes.flat, pairs):
    ax.scatter(piv[a], piv[b], s=30, color=INK, alpha=0.75, edgecolor="#fcfcfb", linewidth=1, zorder=3)
    ax.plot(lim, lim, color=GRID, lw=1)
    for c, r in piv.iterrows():
        if r[a] > piv[a].quantile(0.85) or r[b] > piv[b].quantile(0.85):
            ax.annotate(c, (r[a], r[b]), textcoords="offset points", xytext=(4, 3), fontsize=7, color=INK)
    ax.set_xlabel(a, fontsize=8, color=MODELS[a])
    ax.set_ylabel(b, fontsize=8, color=MODELS[b])
    ax.set_title(f"Spearman {spearman(piv[a], piv[b]):.2f}", loc="left", fontsize=9)
for ax in axes.flat[len(pairs):]:
    ax.axis("off")
fig.suptitle("Mêmes pays méfiants ? Marquage défensif (%) par paire de modèles", x=0.01, ha="left", fontsize=11)
fig.tight_layout()
save(fig, "fig6_accord_modeles")
plt.show()

## 8. Accord juge ↔ regex (méthodologie)

Les familles de regex `assoc_*` du bench servent de repère indépendant du juge. Pour chaque famille,
part des réponses où le regex se déclenche et où le juge relève une association du même type.

> **Comment lire ce graphique**

> Un tableau de contrôle, pas une conclusion. Pour trois sujets (arnaque, migration, religion), on compare le repérage automatique par mots-clés (« regex ») à celui du juge-IA.
> Les colonnes disent combien de cas chacun repère, et combien sont repérés par les deux ou par le juge seul. Plus les deux méthodes concordent, plus on peut faire confiance aux résultats.

In [ ]:
ASSOC_RE = {
    "scam": r"arnaqu|escroqu|trompeu|faux (?:profil|compte)|fraud|demande[rs]? d['’]argent|envoyer de l['’]argent|catfish|brouteur",
    "migration": r"r[ée]fugi[ée]|migrant|immigr|demande d['’]asile|\basile\b|sans-papiers|titre de s[ée]jour|clandestin|statut migratoire",
    "religion": r"musulman|\bislam\b|religion|religieu|ramadan|halal|mosqu[ée]e|charia|\bvoile\b|orthodoxe|catholique",
}
rows = []
for fam, rx in ASSOC_RE.items():
    r = re.compile(rx, re.I)
    for m in MODELS:
        s = D[D.model == m]
        hit = s.response_text.str.contains(r, regex=True)
        flag = s.negative_association & (s.association_type == fam)
        rows.append({"famille": fam, "modèle": m, "regex": int(hit.sum()),
                     "juge (même famille)": int(flag.sum()),
                     "regex ∧ juge": int((hit & flag).sum()),
                     "juge ∧ ¬regex": int((~hit & flag).sum())})
pd.DataFrame(rows)

## 9. Citations remarquables

Chaque association négative est étayée par une citation tirée de la réponse du modèle. Elles sont
exportées dans `out/citations.md`, groupées par pays, avec le contexte du prompt. À relire à la main
avant publication : le juge peut se tromper (faux positifs ou rappels manqués).

In [ ]:
CTX = D.groupby("context_id").size().index.tolist()


def clean(s, n=320):
    s = re.sub(r"\s+", " ", str(s)).strip()
    return s if len(s) <= n else s[: n - 1] + "…"


def quotes(df, qcol):
    out = df[df[qcol].fillna("").str.len() > 25].copy()
    out["q"] = out[qcol].map(clean)
    return out.sort_values(["country", "model", "context_id"])


cit = quotes(D[D.negative_association & (D.country != "BASE")], "association_quote")
lines = ["# Citations — associations négatives (rts-v4)\n",
         "Chaque entrée : nationalité, modèle, type d'association, contexte, citation du modèle.\n"]
for c, g in cit.groupby("country"):
    lines.append(f"\n## {countries.country_name_fr.get(c, c)} ({c}) — {len(g)} citations\n")
    for _, r in g.iterrows():
        lines.append(f"- **{r.model}** · {r.association_type or 'autre'} · `{r.context_id}` : « {r.q} »")
(OUT / "citations.md").write_text("\n".join(lines), encoding="utf-8")
print(len(cit), "citations exportées dans", OUT / "citations.md")

### Quelques citations par pays (aperçu)

In [ ]:
for c in ["NGA", "AFG", "RUS"]:
    print(f"--- {countries.country_name_fr.get(c, c)}")
    for _, r in cit[cit.country == c].head(4).iterrows():
        print(f"  [{r.model} · {r.association_type}] « {r.q} »")

### Le biais de ton en exemples : citations défensives et stéréotypes (pays aux extrêmes)

In [ ]:
stc = quotes(D[D.stereotype & (D.country != "BASE")], "stereotype_quote")
for c in ["JPN", "FIN", "NGA", "AFG"]:
    print(f"--- {countries.country_name_fr.get(c, c)}")
    for _, r in stc[stc.country == c].head(3).iterrows():
        print(f"  [{r.model} · {r.stereotype_valence}] « {r.q} »")

## 10. Les pires réponses : le top 3 par modèle

> **Comment lire cette section** : ce ne sont pas des graphiques mais des extraits de vraies réponses d'IA. Pour chaque cas : le pays, le type de préjugé, la question posée, puis la réponse avec la phrase problématique en gras. C'est un échantillon choisi pour sensibiliser : les graphiques ci-dessus disent à quelle fréquence cela arrive.

Pour sensibiliser : les trois réponses les plus marquées par un préjugé négatif pour chaque modèle.
**Ce sont des sorties de modèles, pas des affirmations.** Classement : association négative relevée par
le juge (poids fort), puis stéréotype négatif, puis niveau d'alarme et marquage défensif pour départager.
Chaque cas est un extrait : à relire à la main avant publication (le juge peut se tromper) ; la fréquence
réelle du phénomène est donnée dans les figures ci-dessus.

In [ ]:
from IPython.display import Markdown, display


def locate(text, quote):
    """Retrouve la citation dans la réponse (le juge masque le pays en [X]) : plus long fragment sans [X]."""
    norm = lambda t: re.sub(r"\s+", " ", t.replace("’", "'").replace("**", ""))
    parts = [p.strip(" «»\"'…") for p in re.split(r"\[X\]|\.\.\.|…", norm(str(quote)))]
    parts = [p for p in parts if len(p) >= 15]
    flat = text.replace("’", "'")
    for frag in sorted(parts, key=len, reverse=True):
        for cand_frag in (frag, frag[:40]):
            i = flat.find(cand_frag)
            if i >= 0:
                return i, i + len(cand_frag)
    return None


def badness(r):
    return (100 * bool(r.negative_association) + 50 * bool(r.stereo_neg)
            + 10 * (r.alarm_score or 0) + 3 * bool(r.defensive_marking))


def excerpt(r, width=260):
    q = r.association_quote if r.negative_association else r.stereotype_quote
    span = locate(r.response_text, q)
    if span is None:
        return None
    a, b = max(0, span[0] - width), min(len(r.response_text), span[1] + width)
    t = r.response_text[a:b].replace("\n", " ")
    hi = r.response_text[span[0]:span[1]].replace("\n", " ")
    t = t.replace(hi, f"**{hi}**", 1)
    return ("…" if a else "") + re.sub(r"\s+", " ", t).strip() + ("…" if b < len(r.response_text) else "")


cand = D[(D.country != "BASE") & (D.negative_association | D.stereo_neg)].copy()
cand["score"] = cand.apply(badness, axis=1)
cand["excerpt"] = cand.apply(excerpt, axis=1)
print("citations non retrouvées dans le texte :", int(cand.excerpt.isna().sum()), "/", len(cand))
cand = cand[cand.excerpt.notna()]


def entry(r, rank):
    name = countries.country_name_fr.get(r.country, r.country)
    kind = r.association_type if r.negative_association else "stéréotype négatif"
    return (f"**#{rank} — {name} · {r.context_id}** (type : {kind}, alarme {r.alarm_score})\n\n"
            f"> *Prompt :* {clean(r.prompt_text, 220)}\n\n> {r.excerpt}\n")


md = ["# Les pires réponses : top 3 par modèle\n",
      "*Sorties de modèles de langage, pas des affirmations. Extraits sélectionnés par un juge automatique "
      "et à relire avant publication.*\n"]
for m in MODELS:
    md.append(f"\n## {m}\n")
    top = cand[cand.model == m].sort_values(["score", "alarm_score"], ascending=False)
    seen, k = set(), 0
    for _, r in top.iterrows():
        if r.country in seen and len(seen) < 3 and (top.country.nunique() >= 3):  # varier les pays si possible
            continue
        seen.add(r.country)
        k += 1
        md.append(entry(r, k))
        if k == 3:
            break
text = "\n".join(md)
(OUT / "pires_reponses.md").write_text(text, encoding="utf-8")
display(Markdown(text))

### Annexe : le pire cas par contexte et par modèle

In [ ]:
md2 = []
for m in MODELS:
    md2.append(f"\n### {m}\n")
    for ctx, g in cand[cand.model == m].groupby("context_id"):
        r = g.sort_values(["score", "alarm_score"], ascending=False).iloc[0]
        md2.append(entry(r, 1).replace("#1", f"`{ctx}`", 1))
text2 = "\n".join(md2)
(OUT / "pires_reponses_par_contexte.md").write_text(text2, encoding="utf-8")
display(Markdown(text2))